# Nectary search: perception and uncertainty-reduction maps

Run in order to train perception, save a checkpoint, and inspect reconstructions,
modality weights, curvature summaries, and UR maps. Training and curvature
sweeps can take time; saved perception weights can be loaded using `chk_path`.

In [ ]:
from pathlib import Path
import os
import sys

PACKAGE_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "NectarySearch").is_dir() and (p / "FlowerTracking").is_dir()),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted code folder.")
NOTEBOOK_DIR = PACKAGE_ROOT / "NectarySearch"
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(NOTEBOOK_DIR)

import math
import random
import numpy as np
import torch
from torch.distributions import ContinuousBernoulli
import matplotlib as mpl
import matplotlib.pyplot as plt
from nectary_env import FlowerParams
from nectary_allocentric_env import AllocentricNectarSearchEnv, CanvasParams
from model import AllocentricNectarySearchModel
from training import train_nectary_search_model, collect_data_random_allocentric
from analysis import run_allocentric_with_ur_heatmaps

mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["axes.spines.top"] = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Environment and perception model

In [ ]:
def compute_entropy(logvar_z):
    return 0.5 * (1 + math.log(2 * torch.pi)) + 0.5 * logvar_z

In [ ]:
flower_params = FlowerParams(
    r0= 2.0,
    R= 50.0,
    L= 25.0,
    c= -1.0,
)

canvas_params = CanvasParams(
    width= 200.0,
    height= 200.0,
    background_value= 0.0,
    flower_value= 1.0,
    hole_value= 0.0,
    pattern_type= "cross",
    pattern_value= 0.5,
    cross_width= 5.0,
    stripe_width= 5.0,
    stripe_offset= 10.0,
    stripe_orientation= "vertical",
    circular_radius= 5.0
)

In [ ]:
chk_path = None

checkpoint = torch.load(chk_path, map_location=DEVICE) if chk_path else None

if not checkpoint:
    checkpoint = {
        "flower_pattern": "cross",
        "visual_field_size": 25,
        "mech_embedding_dim": 16,
        "vis_embedding_dim": 32,
        "fusion_dim": 256,
        "latent_dim": 512,
        "hidden_dim": 256,
        "visual_likelihood": "gaussian",
        "posterior_fusion": "moe",
        "rnn_type": "rnn",
        "learn_decoder_variance": False,
        "modality_rnn_mode": "separate",
        "moment_matching": True
    }

In [ ]:
""" Build the nectary search environment """

c_vals = [-1.0, -2.0, -20.0, -100.]

training_envs = []

for c in c_vals:
    canvas_params.pattern_type = checkpoint['flower_pattern']
    training_envs.append(
        AllocentricNectarSearchEnv(
            flower=FlowerParams(r0=flower_params.r0, R=flower_params.R, L=flower_params.L, c=c),
            canvas=canvas_params,
            dt=0.01,
            max_steps=100,
            max_press_depth=5,
            visual_field_size=checkpoint['visual_field_size']
        )
    )

## Example training trajectory

In [ ]:
xm, xv, a, _ = collect_data_random_allocentric(training_envs[0], 1, 300)
positions = a[0].cpu().numpy().copy()
positions[:, 0] *= training_envs[0].canvas.width
positions[:, 1] *= training_envs[0].canvas.height
positions[:, 2] *= training_envs[0].canvas.width
positions[:, 3] *= training_envs[0].canvas.height

plt.imshow(training_envs[0].canvas_image, cmap='gray')
plt.plot(positions[:, 2], positions[:, 3], 'b-o', markersize=2)
plt.show()

In [ ]:
""" Build the nectary search model """
model = AllocentricNectarySearchModel(
    mech_obs_dim=training_envs[0].observation_space['mech'].shape[0],
    vis_obs_shape=(1,) + training_envs[0].observation_space['vision'].shape,
    action_dim=training_envs[0].action_space.shape[0],
    mech_embedding_dim=checkpoint['mech_embedding_dim'],
    vis_embedding_dim=checkpoint['vis_embedding_dim'],
    fusion_dim=checkpoint['fusion_dim'],
    latent_dim=checkpoint['latent_dim'],
    hidden_dim=checkpoint['hidden_dim'],
    visual_likelihood=checkpoint['visual_likelihood'],
    posterior_fusion=checkpoint['posterior_fusion'],
    rnn_type=checkpoint['rnn_type'],
    modality_rnn_mode=checkpoint.get('modality_rnn_mode'),
    beta=0.002,
    learn_decoder_variance=checkpoint.get('learn_decoder_variance', True),
    moment_matching=checkpoint.get('moment_matching')
).to(DEVICE)

## Train and save perception

In [ ]:
TRAINING_SETTINGS = {
    "num_epochs": 400,
    "batch_size": 40,
    "seq_len": training_envs[0].max_steps,
    "learning_rate": 7e-4,
    "print_every": 10,
    "allocentric": True,
    "min_depth": 0.1,
    "randomize_visual_pattern_each_epoch": True,
    "randomize_visual_colors_each_epoch": True,
}
if chk_path is None:
    model = train_nectary_search_model(training_envs, model, **TRAINING_SETTINGS)
else:
    model.load_state_dict(checkpoint["model_state_dict"])

In [ ]:
CHECKPOINT_PATH = Path("checkpoints/perception.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
checkpoint["model_state_dict"] = model.state_dict()
checkpoint["beta"] = model.beta
if chk_path is None:
    checkpoint["num_epochs"] = TRAINING_SETTINGS["num_epochs"]
    checkpoint["lr"] = TRAINING_SETTINGS["learning_rate"]
    checkpoint["batch_size"] = TRAINING_SETTINGS["batch_size"]
    checkpoint["min_depth"] = TRAINING_SETTINGS["min_depth"]
    checkpoint["perception_training"] = dict(TRAINING_SETTINGS)
torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Saved perception checkpoint to {CHECKPOINT_PATH}")

## Reconstructions, modality weights, and uncertainty

In [ ]:
""" Visualize responses to example episodes """
model.to(DEVICE)

env_idx = 0
training_envs[env_idx].canvas.flower_value = 1.0
training_envs[env_idx].canvas.pattern_value = 0.5

X_mech, X_vis, A, xyd = collect_data_random_allocentric(training_envs[env_idx], batch_size=20, seq_len=training_envs[0].max_steps, min_depth=0.5)
X_mech = X_mech.to(DEVICE)
X_vis = X_vis.to(DEVICE)
A = A.to(DEVICE)
test_responses, test_losses = model(X_mech, X_vis, A)

In [ ]:
ind = 2
obs_names = ['Contact', '$f_n$', '$n_x$', '$n_y$', 'Slip']
fig = plt.figure(figsize=(12, 6))
for i, obs in enumerate(obs_names):
    xhat = test_responses['x_mech_hat'][ind, :, i].detach().cpu()
    if obs == 'Contact':
        xhat = torch.sigmoid(xhat)

    plt.subplot(3, 3, i+1)
    plt.plot(X_mech[ind, :, i].cpu().numpy(), label='True')
    plt.plot(xhat.numpy(), 'r-', label='Reconstructed')
    plt.title(obs, fontsize=10)

fig.legend(*plt.gca().get_legend_handles_labels(), bbox_to_anchor=(0.9, 0.5), frameon=False)

plt.tight_layout()

plt.show()

In [ ]:
plt.plot(test_responses['moe_weights'][ind][:, 0].detach().cpu().numpy(), label='mech weight')
plt.plot(test_responses['moe_weights'][ind][:, 1].detach().cpu().numpy(), label='visual weight')
plt.legend()

In [ ]:
i = 0

if model.visual_likelihood == 'gaussian':
    rec = test_responses['x_vis_hat'][ind][i]
else:
    rec = ContinuousBernoulli(probs=test_responses['x_vis_hat'][ind][i]).mean

plt.subplot(1, 2, 1)
plt.imshow(rec.reshape(*training_envs[env_idx].visual_patch_shape).cpu().detach().numpy(), cmap='gray', vmin=0, vmax=1)
plt.subplot(1, 2, 2)
plt.imshow(X_vis[ind][i].reshape(*training_envs[env_idx].visual_patch_shape).cpu().detach().numpy(), cmap='gray', vmin=0, vmax=1)
plt.show()

In [ ]:
mech_mu = test_responses['mech_post_mu']
vis_mu = test_responses['vis_post_mu']
disag = ((mech_mu - vis_mu)**2).sum(-1).sqrt()

plt.plot(disag.mean(0).detach().cpu().numpy())
plt.title("Unimodal posterior disagreement")
plt.show()

In [ ]:
prior_entropy = compute_entropy(test_responses['prior_logvar'])
post_entropy = compute_entropy(test_responses['post_logvar'])
entropy_red = (prior_entropy - post_entropy).sum(-1)

plt.plot(entropy_red.mean(0).detach().cpu().numpy())
plt.title("Average uncertainty reduction")
plt.show()

## Curvature and visual-pattern summaries

In [ ]:
device = 'cpu'
model.to(device)
def avg_entropy_reduction_vs_c(pattern='none'):
    c_sweep = -np.linspace(1, 50, 5)
    b = 200
    avg_entropies = {'vis': [], 'mech': [], 'joint': [], 'weighted': []}
    sem_entropies = {'vis': [], 'mech': [], 'joint': [], 'weighted': []}
    for c in c_sweep:

        canvas_params.pattern_type = pattern
        canvas_params.pattern_value = 0.5
        canvas_params.flower_value = 1.0

        env = AllocentricNectarSearchEnv(
                flower=FlowerParams(r0=flower_params.r0, R=flower_params.R, L=flower_params.L, c=c),
                canvas=canvas_params,
                dt=0.01,
                max_steps=100,
                max_press_depth=5,
                visual_field_size=checkpoint['visual_field_size']
            )

        X_mech, X_vis, A, xyd = collect_data_random_allocentric(env, batch_size=b, seq_len=env.max_steps, min_depth=0.5, trajectory_mode='random')
        X_mech, X_vis = X_mech.to(device), X_vis.to(device)
        A = A.to(device)
        test_responses, test_losses = model(X_mech, X_vis, A)

        prior_entropy = compute_entropy(test_responses['prior_logvar'])

        for k, v in zip(['vis', 'mech', 'joint', 'weighted'], ['vis_post_logvar', 'mech_post_logvar', 'post_logvar', '']):
            if k == 'weighted':
                weighted_mech_post = test_responses['moe_weights'][..., [0]] * test_responses['mech_post_logvar']
                weighted_vis_post = test_responses['moe_weights'][..., [1]] * test_responses['vis_post_logvar']
                post_entropy = compute_entropy(weighted_mech_post + weighted_vis_post)
            else:
                post_entropy = compute_entropy(test_responses[v])
            entropy = prior_entropy - post_entropy

            mean_ent = entropy.sum(-1).mean()
            sem_ent = entropy.sum(-1).std() / math.sqrt(b * env.max_steps)

            avg_entropies[k].append(mean_ent.detach().cpu().numpy())
            sem_entropies[k].append(sem_ent.detach().cpu().numpy())

    return c_sweep, avg_entropies, sem_entropies

In [ ]:
c_sweep, avg_entropies_cross, sem_entropies_cross = avg_entropy_reduction_vs_c('cross')

In [ ]:
_, avg_entropies_none, sem_entropies_none = avg_entropy_reduction_vs_c('none')

In [ ]:
_, avg_entropies_stripes, sem_entropies_stripes = avg_entropy_reduction_vs_c('stripes')

In [ ]:
_, avg_entropies_circ, sem_entropies_circ = avg_entropy_reduction_vs_c('circular')

In [ ]:
fig = plt.figure(figsize=(18, 4))
for i, k in enumerate(['vis', 'mech', 'joint', 'weighted']):
    plt.subplot(1, 4, i+1)

    plt.errorbar(c_sweep, avg_entropies_cross[k], yerr=sem_entropies_cross[k], label='cross')
    plt.errorbar(c_sweep, avg_entropies_none[k], yerr=sem_entropies_none[k], c='orange', label='no pattern')
    plt.errorbar(c_sweep, avg_entropies_stripes[k], yerr=sem_entropies_stripes[k], c='magenta', label='stripes')
    plt.errorbar(c_sweep, avg_entropies_circ[k], yerr=sem_entropies_circ[k], c='purple', label='circular')

    plt.xlabel('$c$')
    plt.ylabel('Average Entropy Reduction')
    plt.title({"vis": "Visual", "mech": "Mechanosensory", "joint": "Aggregate entropy", "weighted": "Weighted-unimodal"}[k])

plt.legend()

plt.show()

## UR maps along radial and circular trajectories

In [ ]:
class URMap:

    @torch.no_grad()
    def __init__(self, model, c=-1.0, angle=0., r=0.95, path='radial', expected=False, modality='joint'):

        self.model = model

        canvas_params.pattern_type = 'cross'
        canvas_params.pattern_value = 0.5
        canvas_params.flower_value = 1.0
        env = AllocentricNectarSearchEnv(
                flower=FlowerParams(r0=flower_params.r0, R=flower_params.R, L=flower_params.L, c=c),
                canvas=canvas_params,
                dt=0.01,
                max_steps=21,
                max_press_depth=5,
                visual_field_size=checkpoint['visual_field_size']
            )

        self.env = env

        self.H, self.times, self.traj_xy, self.traj_uncertainties, self.traj_obs, (self.X, self.Y, self.valid_mask) = run_allocentric_with_ur_heatmaps(
            self.env, self.model, compute_heatmap_every=4, angle=angle, end_r=2., depth_fixed=0.5,
            path=path,
            start_r_frac=r,
            radial_step_mm=1.0,
            angle_step_rad=0.1*np.pi,
            mask_to_flower=False,
            expected=expected,
            device='cpu',
            blank_visual=False,
            blank_mechanosensory=False,
            modality=modality
        )

In [ ]:
radial_funnel_ur = URMap(model=model, c=-1.0, angle=np.pi, expected=False, modality='joint')
radial_flat_ur = URMap(model=model, c=-1000.0, angle=np.pi, expected=False, modality='joint')

In [ ]:
circ_funnel_ur = URMap(model=model, c=-1.0, path='circular', r=0.5, expected=False, modality='joint')
circ_flat_ur = URMap(model=model, c=-1000.0, path='circular', r=0.5, expected=False, modality='joint')

In [ ]:
maps = [radial_funnel_ur, radial_flat_ur, circ_funnel_ur, circ_flat_ur]
titles = ["Radial (Funnel)", "Radial (Flat)", "Circular (Funnel)", "Circular (Flat)"]
fig, ax = plt.subplots()
for result, title in zip(maps, titles):
    ax.plot(result.times, result.H.nanmean((-1, -2)).cpu().numpy(), label=title)
ax.set_xlabel("Observation step")
ax.set_ylabel("Mean map UR (nats)")
ax.legend()
vmn, vmx = ax.get_ylim()
plt.show()

In [ ]:
for result, title in zip(maps, titles):
    n_maps = len(result.times)
    fig, axes = plt.subplots(1, n_maps, figsize=(18, 4), squeeze=False)
    X_local = result.X - result.env.flower_center_xy[0]
    Y_local = result.Y - result.env.flower_center_xy[1]
    for k, observation_step in enumerate(result.times):
        ax = axes[0, k]
        ax.pcolormesh(X_local, Y_local, result.H[k].detach().cpu().numpy(),
                      shading="nearest", cmap="magma", vmin=vmn-50, vmax=vmx+50)
        x, y = result.traj_xy[observation_step]
        ax.plot(x, y, "ro", markersize=4)
        ax.set_title(f"Step {observation_step}")
        ax.set_aspect("equal")
        ax.axis("off")
    fig.suptitle(title, fontsize=18)
    fig.tight_layout()
    plt.show()